# Final Task-Type Detector (LOSO)
This notebook runs a leave-one-subject-out (LOSO) training pipeline over tabular features,
exports per-fold metrics, per-class reports and confusion matrices, computes simple aggregated
summary statistics (mean/std) and trains & saves a final model on the full dataset. Run cells sequentially inside the project venv.

In [ ]:
# Config and imports
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (f1_score, balanced_accuracy_score, accuracy_score, classification_report, confusion_matrix)
import joblib
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
# Paths (adjust if needed)
DATASET = Path('/home/g0amer/Desktop/thesis/research_outputs/fusion/v1_fusion/fusion_dataset.csv')
OUT_ROOT = Path('/home/g0amer/Desktop/thesis/research_outputs/fusion_training/final_detector_notebook')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

In [ ]:
# Quick dataset inspection
df = pd.read_csv(DATASET, low_memory=False)
print('rows', len(df), 'columns', len(df.columns))
print('labels counts:')
print(df['pseudo_label'].value_counts())
print('unique subjects', df['subject_id'].nunique())

In [ ]:
# Helper utilities
def get_feature_columns(df):
    exclude = {"subject_id", "task_name", "task_file", "split", "window_idx",
               "start_idx", "end_idx", "n_samples", "pseudo_label",
               "eeg_features_5s__split", "n_modalities_present"}
    cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    return cols

def build_classifiers():
    cls = {
        'logreg': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
        'rf': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE),
        'mlp': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=RANDOM_STATE),
    }
    if HAS_XGB:
        cls['xgboost'] = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=RANDOM_STATE)
    return cls

In [ ]:
# LOSO training function (writes per-fold artifacts)
def run_loso(dataset_path, out_root):
    df = pd.read_csv(dataset_path, low_memory=False)
    if 'pseudo_label' not in df.columns:
        raise RuntimeError('expected column pseudo_label in dataset')

    subjects = sorted(df['subject_id'].unique())
    feature_cols = get_feature_columns(df)
    print(f'Found {len(subjects)} subjects, {len(feature_cols)} features')

    classifiers = build_classifiers()
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    summary_rows = []

    for name, clf in classifiers.items():
        print('Running LOSO for', name)
        model_dir = out_root / name
        model_dir.mkdir(parents=True, exist_ok=True)
        per_fold_dir = model_dir / 'folds'
        per_fold_dir.mkdir(exist_ok=True)

        fold_metrics = []

        for subj in subjects:
            train = df[df['subject_id'] != subj]
            test = df[df['subject_id'] == subj]
            if test.empty:
                continue

            X_train = train[feature_cols].values
            y_train = train['pseudo_label'].values
            X_test = test[feature_cols].values
            y_test = test['pseudo_label'].values

            pipe = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), clf)
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)

            m = {
                'model': name,
                'subject_id': int(subj),
                'n_test': int(len(y_test)),
                'accuracy': float(accuracy_score(y_test, y_pred)),
                'macro_f1': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
                'balanced_acc': float(balanced_accuracy_score(y_test, y_pred)),
            }

            report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            labels_sorted = sorted(np.unique(df['pseudo_label']))
            cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

            pd.DataFrame([m]).to_csv(per_fold_dir / f'fold_{int(subj)}.csv', index=False)
            pd.DataFrame(report).T.to_csv(per_fold_dir / f'fold_{int(subj)}_per_class.csv')
            pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted).to_csv(per_fold_dir / f'fold_{int(subj)}_confusion.csv')

            fold_metrics.append(m)

        pd.DataFrame(fold_metrics).to_csv(model_dir / 'fold_metrics.csv', index=False)

        # train final model on full dataset and save
        full_pipe = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), clf)
        full_pipe.fit(df[feature_cols].values, df['pseudo_label'].values)
        joblib.dump(full_pipe, model_dir / 'final_model.joblib')

        if fold_metrics:
            dfm = pd.DataFrame(fold_metrics)
            summary_rows.append({
                'model': name,
                'n_folds': int(len(dfm)),
                'macro_f1_mean': float(dfm['macro_f1'].mean()),
                'macro_f1_std': float(dfm['macro_f1'].std()),
                'balanced_acc_mean': float(dfm['balanced_acc'].mean()),
                'balanced_acc_std': float(dfm['balanced_acc'].std()),
            })

    pd.DataFrame(summary_rows).to_csv(out_root / 'benchmark_summary_from_loso.csv', index=False)
    print('LOSO complete. Outputs under', out_root)

# End run_loso

In [ ]:
# Execute LOSO (this may take a while)
run_loso(DATASET, OUT_ROOT)

In [ ]:
# Aggregate per-fold metrics into a simple summary (mean/std), no CIs
summary = []
for model_dir in sorted(OUT_ROOT.iterdir()):
    if not model_dir.is_dir():
        continue
    fm = model_dir / 'fold_metrics.csv'
    if not fm.exists():
        continue
    dfm = pd.read_csv(fm)
    if dfm.empty:
        continue
    summary.append({
        'model': model_dir.name,
        'n_folds': int(len(dfm)),
        'macro_f1_mean': float(dfm['macro_f1'].mean()),
        'macro_f1_std': float(dfm['macro_f1'].std()),
        'balanced_acc_mean': float(dfm['balanced_acc'].mean()),
        'balanced_acc_std': float(dfm['balanced_acc'].std()),
    })
pd.DataFrame(summary).to_csv(OUT_ROOT / 'benchmark_summary_validated_from_notebook.csv', index=False)
print('Wrote', OUT_ROOT / 'benchmark_summary_validated_from_notebook.csv')

In [ ]:
# Load best model and run a small inference sanity-check
bm = pd.read_csv(OUT_ROOT / 'benchmark_summary_validated_from_notebook.csv')
bm = bm.sort_values('macro_f1_mean', ascending=False)
print('model ranking:')
print(bm[['model','macro_f1_mean']])
best = bm.iloc[0]['model']
print('best model', best)
model_path = OUT_ROOT / best / 'final_model.joblib'
import joblib
m = joblib.load(model_path)
# example: predict first 5 rows
feature_cols = get_feature_columns(df)
X = df[feature_cols].iloc[:5].values
print('example predictions:', m.predict(X))

## Next steps
- Optionally convert the best `final_model.joblib` to ONNX for deployment.
- Export figures (per-class heatmap, macro-F1 CI bar) from the saved per-fold artifacts.
- If you want, I can run this notebook now and return the produced `benchmark_summary_validated_from_notebook.csv` and the saved model path.